In [ ]:
import torch
import torch.nn as nn
from vae import load_trained_improved_vae

In [ ]:
vae, _ = load_trained_improved_vae(
    checkpoint_path="/home/fer/Escritorio/dragons/dragon/vae/VAE_TOTAL_7/improved_vae_best_epoch_750.pth",
    latent_dim=1024,
    device="cpu",
)


vae.eval()


class DecoderOnly(nn.Module):
    def __init__(self, decoder_input, decoder):
        super().__init__()
        self.decoder_input = decoder_input
        self.decoder = decoder

    def forward(self, z):
        # Linear projection
        x = self.decoder_input(z)
        # Reshape to [batch, 2048, 4, 4]
        x = x.view(-1, 2048, 4, 4)
        # Pass through ConvTranspose decoder
        return self.decoder(x)


# Extract decoder

decoder_only = DecoderOnly(vae.decoder_input, vae.decoder)
decoder_only.eval()


latent_dim = 1024
dummy_latent = torch.randn(1, latent_dim)

# Export to ONNX
torch.onnx.export(
    decoder_only,
    dummy_latent,
    "vae_decoder.onnx",
    input_names=["latent"],
    output_names=["reconstructed"],
    dynamic_axes={"latent": {0: "batch_size"}, "reconstructed": {0: "batch_size"}},
    opset_version=17,  # Use a high enough opset for better compatibility
)

print("✅ Decoder exported to vae_decoder.onnx")